# There Is No Gap: A Reader Knows Exactly the Loops It Has Closed

### Proving "the law is the address, the variable is the location" from one bit upward

**Dean Kulik — QuHarmonics Research Group**
CC BY-NC 4.0

---

## The claim, in a form that can fail

The statement under test: *the law is the address; the variable is the location; there is no gap.* The method: *everything is a reader and being read; start at the smallest scale and the largest gap.*

Take systems whose parts read each other through signed links: each part reads its neighbour either as it is (+) or inverted (−), and is read in turn. Then the statement becomes three claims that can each come out wrong.

- **(A) The law is the address.** The dynamics, up to a relabelling of how each part is read (a *gauge*), is fixed by address-level bits alone: the sign products around loops. States, gains and delays are locations inside that law.
- **(B) There is no gap in the law.** What looked like a local→global gap lives entirely in partial readers. A reader that has read a set of links $S$ knows exactly $\beta_1(S)$ bits of the global structure: one bit per independent loop it has closed, and nothing before it closes one.
- **(C) The largest gap is one loop wide.** On a ring, a reader that has read every link but one, or watched one node for one step less than a full circuit, knows nothing. Closing the loop gives everything, and the system closes its own loop on every circuit.

**What would falsify it:** two sign patterns with the same loop bits but different dynamics; a reader of a loop-free set of links that knows anything; a continuous ring whose long-run behaviour disagrees with (loop sign, location threshold).

---

In [1]:
import numpy as np, itertools, math
from collections import Counter

# ---------- 1. one bit reading itself
MAPS = {"identity": (0, 1), "negation": (1, 0), "const 0": (0, 0), "const 1": (1, 1)}
def orbit_lengths(m):
    out = []
    for x0 in (0, 1):
        seen, x = [], x0
        while x not in seen: seen.append(x); x = m[x]
        out.append(len(seen) - seen.index(x))
    return out

# ---------- 2. signed Boolean ring: x_i <- x_{i-1} XOR s_i  (s_i = 1 means "negate")
def ring_perm(s):
    N = len(s); n = 1 << N; X = np.arange(n)
    bits = (X[:, None] >> np.arange(N)) & 1
    new = np.roll(bits, 1, axis=1) ^ np.array(s)
    return (new << np.arange(N)).sum(1)
def cycle_type(perm):
    seen = np.zeros(len(perm), bool); c = Counter()
    for x in range(len(perm)):
        if seen[x]: continue
        L = 0; y = x
        while not seen[y]: seen[y] = True; y = perm[y]; L += 1
        c[L] += 1
    return tuple(sorted(c.items()))
def switch(s, g):
    N = len(s); return tuple(s[i] ^ g[i] ^ g[i - 1] for i in range(N))
def gauge_between(s, t):
    N = len(s); g = [0] * N
    for i in range(1, N): g[i] = g[i - 1] ^ s[i] ^ t[i]
    return tuple(g) if switch(s, g) == tuple(t) else None

# ---------- 3. information a partial reader has about the loop bit
def H(counter):
    n = sum(counter.values()); return -sum(v / n * math.log2(v / n) for v in counter.values() if v)
def mutual_info(pairs):
    pairs = list(pairs); a = Counter(p[0] for p in pairs); b = Counter(p[1] for p in pairs); ab = Counter(pairs)
    return H(a) + H(b) - H(ab)
def spatial_mi(N, S):
    return mutual_info((tuple(s[i] for i in S), sum(s) % 2) for s in itertools.product((0, 1), repeat=N))
def temporal_mi(N, k, node=0):
    pairs = []
    for s in itertools.product((0, 1), repeat=N):
        perm = ring_perm(s)
        for x in range(1 << N):
            seq, y = [], x
            for _ in range(k + 1): seq.append((y >> node) & 1); y = perm[y]
            pairs.append((tuple(seq), sum(s) % 2))
    return mutual_info(pairs)

# ---------- 4. continuous ring  x_i' = -x_i + sigma_i tanh(G x_{i-1}), delay tau on the link into node 0
def Gcrit(N, tau):
    """per-stage gain at which a negative loop's origin loses stability: N arccos(1/G) + tau sqrt(G^2-1) = pi"""
    f = lambda G: N * math.acos(1 / G) + tau * math.sqrt(G * G - 1) - math.pi
    if f(1e6) < 0: return math.inf
    lo, hi = 1.0 + 1e-12, 2.0
    while f(hi) < 0: hi *= 2
    for _ in range(200):
        mid = 0.5 * (lo + hi); (lo, hi) = (mid, hi) if f(mid) < 0 else (lo, mid)
    return 0.5 * (lo + hi)
def ring_run(sig, G, X0, tau, T=(500.0, 1000.0), dt=0.02, record_final=False):
    """sig, G: (B,N),(B,) ; X0: (B,N) constant initial history. Returns swings (len(T),B)."""
    B, N = X0.shape; x = X0.copy(); lag = int(round(tau / dt)); L = lag + 1
    buf = np.zeros((L, B)); buf[:] = X0[:, -1]              # history of x_{N-1}
    st = [int(t / dt) for t in T]; w0 = [int(0.9 * s) for s in st]
    mx = [np.full((B, N), -np.inf) for _ in T]; mn = [np.full((B, N), np.inf) for _ in T]
    Gc = G[:, None]
    def rhs(x, xd_last):
        inp = np.roll(x, 1, axis=1); inp[:, 0] = xd_last
        return -x + sig * np.tanh(Gc * inp)
    for k in range(st[-1]):
        xd = buf[(k - lag) % L] if lag > 0 else x[:, -1]
        if lag > 0:
            k1 = rhs(x, xd); k2 = rhs(x + 0.5*dt*k1, xd); k3 = rhs(x + 0.5*dt*k2, xd); k4 = rhs(x + dt*k3, xd)
        else:
            f = lambda y: -y + sig * np.tanh(Gc * np.roll(y, 1, axis=1))
            k1 = f(x); k2 = f(x + 0.5*dt*k1); k3 = f(x + 0.5*dt*k2); k4 = f(x + dt*k3)
        x = x + dt / 6 * (k1 + 2*k2 + 2*k3 + k4)
        if lag > 0: buf[(k + 1) % L] = x[:, -1]
        for h in range(len(T)):
            if w0[h] <= k + 1 <= st[h]:
                np.maximum(mx[h], x, out=mx[h]); np.minimum(mn[h], x, out=mn[h])
    sw = np.array([(mx[h] - mn[h]).max(1) for h in range(len(T))])
    return (sw, x) if record_final else sw

# ---------- 5. signed graphs: what a reader of a link set S knows about the gauge class
def beta1(edges, nodes):
    parent = {v: v for v in nodes}
    def find(v):
        while parent[v] != v: parent[v] = parent[parent[v]]; v = parent[v]
        return v
    comps = len(nodes)
    for a, b in edges:
        ra, rb = find(a), find(b)
        if ra != rb: parent[ra] = rb; comps -= 1
    return len(edges) - len(nodes) + comps
def gauge_class(edges, nodes, s):
    """canonical form: switch so spanning-forest edges are positive; return signs on the remaining edges"""
    adj = {v: [] for v in nodes}
    for j, (a, b) in enumerate(edges): adj[a].append((b, j)); adj[b].append((a, j))
    g, tree = {}, set()
    for r in nodes:
        if r in g: continue
        g[r] = 0; stack = [r]
        while stack:
            u = stack.pop()
            for v, j in adj[u]:
                if v not in g: g[v] = g[u] ^ s[j]; tree.add(j); stack.append(v)
    return tuple(s[j] ^ g[a] ^ g[b] for j, (a, b) in enumerate(edges) if j not in tree)

## Step 1 — The smallest scale: one bit reading itself

A single bit that reads itself has four possible laws. List them all and ask which have a fixed point.

In [2]:
print(f"{'law':>10} | {'orbit length from 0, from 1':>28} | fixed point?")
print("-" * 58)
for name, m in MAPS.items():
    L = orbit_lengths(m)
    print(f"{name:>10} | {str(L):>28} | {'yes' if 1 in L else 'NO'}")

       law |  orbit length from 0, from 1 | fixed point?
----------------------------------------------------------
  identity |                       [1, 1] | yes
  negation |                       [2, 2] | NO
   const 0 |                       [1, 1] | yes
   const 1 |                       [1, 1] | yes


Negation is the only law on one bit with no fixed point, so "everything must change" (C1) at the smallest scale *is* the negative self-loop, and its orbit is a 2-cycle: the smallest clock. Identity is the smallest memory. Which one you have is one bit of address, the sign of the self-read. The state, 0 or 1, is the location, and it does not change the answer.

---
## Step 2 — A ring of readers

Now $N$ bits, each reading its predecessor, $x_i \leftarrow x_{i-1} \oplus s_i$ ($s_i = 1$ means "read inverted"). Every node reads and is read. There are $2^N$ sign patterns. If only the address matters, the patterns should fall into exactly two dynamics, split by the loop bit $\bigoplus_i s_i$, and any two patterns with the same loop bit should be the same law seen through a relabelling $g$: $F_{s'}(x \oplus g) = F_s(x) \oplus g$.

In [3]:
print(f"{'N':>3} {'patterns':>9} {'dynamics (loop +, loop -)':>26} {'same-bit pairs conjugate':>26} {'cross-bit pairs with a gauge':>29}")
print("-" * 98)
for N in range(1, 11):
    pats = list(itertools.product((0, 1), repeat=N))
    types = {0: set(), 1: set()}
    for s in pats:
        types[sum(s) % 2].add(cycle_type(ring_perm(s)))
    conj = total = cross = 0
    if N <= 8:
        ref = {0: tuple([0] * N), 1: tuple([1] + [0] * (N - 1))}
        for s in pats:
            p = sum(s) % 2
            g = gauge_between(s, ref[p]); total += 1
            if g is not None:
                gm = sum(b << i for i, b in enumerate(g))
                ps, pr = ring_perm(s), ring_perm(ref[p])
                X = np.arange(1 << N)
                conj += bool(np.all(pr[X ^ gm] == (ps[X] ^ gm)))
            cross += gauge_between(s, ref[1 - p]) is not None
    note = f"{conj}/{total}" if N <= 8 else "(not run)"
    print(f"{N:>3} {len(pats):>9} {str((len(types[0]), len(types[1]))):>26} {note:>26} {str(cross) if N <= 8 else '':>29}")
distinct = all(len({cycle_type(ring_perm(s)) for s in [tuple([0]*N), tuple([1]+[0]*(N-1))]}) == 2 for N in range(1, 11))
print(f"\nloop + and loop - give different dynamics for every N: {distinct}")

  N  patterns  dynamics (loop +, loop -)   same-bit pairs conjugate  cross-bit pairs with a gauge
--------------------------------------------------------------------------------------------------
  1         2                     (1, 1)                        2/2                             0
  2         4                     (1, 1)                        4/4                             0
  3         8                     (1, 1)                        8/8                             0
  4        16                     (1, 1)                      16/16                             0
  5        32                     (1, 1)                      32/32                             0
  6        64                     (1, 1)                      64/64                             0
  7       128                     (1, 1)                    128/128                             0
  8       256                     (1, 1)                    256/256                             0
  9       512      

For every ring size, the $2^N$ ways of wiring the reads collapse to exactly two laws, one per loop bit. Every pattern is the other same-bit patterns seen through a relabelling, and no relabelling crosses the loop bit. An individual link's sign is a reader's convention; only the loop's sign is law. That is claim (A) at the discrete level.

---
## Step 3 — The largest gap

How much does a partial reader know about the loop bit? Two kinds of reader, with the sign pattern and the starting state unknown and uniform:

- a **spatial** reader that has read the signs on some set of links $S$;
- a **temporal** reader that watches one node for $k$ steps.

In [4]:
print("Spatial reader: bits known about the loop bit, after reading |S| of N links (max over all S of that size)")
for N in range(2, 11):
    row = []
    for r in range(N + 1):
        row.append(max(spatial_mi(N, S) for S in itertools.combinations(range(N), r)))
    print(f"  N={N:>2}: " + "  ".join(f"{v:.0f}" for v in row))
print("\nTemporal reader: bits known after watching one node for k steps (k = 0 .. N+1)")
for N in range(2, 9):
    print(f"  N={N:>2}: " + "  ".join(f"{temporal_mi(N, k):.0f}" for k in range(N + 2)))

Spatial reader: bits known about the loop bit, after reading |S| of N links (max over all S of that size)
  N= 2: 0  0  1
  N= 3: 0  0  0  1
  N= 4: 0  0  0  0  1
  N= 5: 0  0  0  0  0  1
  N= 6: 0  0  0  0  0  0  1
  N= 7: 0  0  0  0  0  0  0  1
  N= 8: 0  0  0  0  0  0  0  0  1
  N= 9: 0  0  0  0  0  0  0  0  0  1
  N=10: 0  0  0  0  0  0  0  0  0  0  1

Temporal reader: bits known after watching one node for k steps (k = 0 .. N+1)
  N= 2: 0  0  1  1
  N= 3: 0  0  0  1  1
  N= 4: 0  0  0  0  1  1
  N= 5: 0  0  0  0  0  1  1
  N= 6: 0  0  0  0  0  0  1  1
  N= 7: 0  0  0  0  0  0  0  1  1
  N= 8: 0  0  0  0  0  0  0  0  1  1


Both readers hit the same cliff. Read any $N-1$ of the $N$ links, or watch a node for $N-1$ steps, and the reader knows exactly **zero** bits about the law. Read the last link, or wait the last step, and it knows the whole bit. This is the largest possible gap: everything but one location visited, and nothing known.

It is also why the gap is not in the law. The temporal reader succeeds for a structural reason, $x_i(t+N) = x_i(t) \oplus \bigoplus_j s_j$: after one circuit, every node is read back to itself, through the whole loop. The system reads its own address on every circuit. Only a reader that has not closed the loop is missing anything. That is claim (C).

---
## Step 4 — The same law in continuous time, with delay

Now a continuous ring, $\dot x_i = -x_i + \sigma_i \tanh(G\,x_{i-1})$ with $\sigma_i = \pm 1$, and a delay $\tau$ on the link into node 0. Since $\tanh$ is odd, flipping how a node is read is an exact relabelling, so the discrete result should carry over. The prediction uses only the address bit $P = \prod_i \sigma_i$ plus one location threshold:

- $P = +1$: no persistent oscillation at any gain or delay (the latch or rest corner, $\Phi = 0$);
- $P = -1$: persistent oscillation exactly when $G > G_c(N, \tau)$, where $N\arccos(1/G) + \tau\sqrt{G^2-1} = \pi$ (the clock corner).

Each run is judged at two horizons, $T = 500$ and $T = 1000$, so a slow transient cannot pass for a cycle. Gains within 8% of $G_c$ (or 5% of 1 on positive loops) are excluded as knife edges.

In [5]:
rng = np.random.default_rng(7)
N, B, tau = 5, 6, 1.0
sig = rng.choice([-1.0, 1.0], size=(B, N)); G = rng.uniform(0.5, 4, B); X0 = rng.uniform(-1, 1, (B, N))
g = rng.choice([-1.0, 1.0], size=(B, N)); sig2 = sig * g * np.roll(g, 1, axis=1)
_, xa = ring_run(sig, G, X0, tau, T=(20.0, 40.0), record_final=True)
_, xb = ring_run(sig2, G, X0 * g, tau, T=(20.0, 40.0), record_final=True)
print(f"relabelled ring vs original, max |x' - g x| after t = 40: {float(np.abs(xb - g * xa).max()):.1e}\n")
print(f"{'N':>2} {'tau':>4} {'G_c':>8} {'agree':>7} {'loop+ osc @500':>15} {'loop+ osc @1000':>16} {'loop- osc':>10}")
print("-" * 70)
tot = n_all = 0
for N in range(1, 7):
    for tau in (0.0, 1.0):
        B = 24
        P = np.array([1] * (B // 2) + [-1] * (B // 2))
        sig = rng.choice([-1.0, 1.0], size=(B, N)); sig[:, 0] *= P * np.prod(sig, axis=1)
        gc = Gcrit(N, tau); G = rng.uniform(0.5, 4.0, B)
        for b in range(B):
            while (P[b] < 0 and abs(G[b] / gc - 1) < 0.08) or (P[b] > 0 and abs(G[b] - 1) < 0.05):
                G[b] = rng.uniform(0.5, 4.0)
        X0 = rng.uniform(-1, 1, (B, N))
        sw = ring_run(sig, G, X0, tau)
        pred = (P < 0) & (G > gc); osc = sw[-1] > 1e-3
        a = int((pred == osc).sum()); tot += a; n_all += B
        print(f"{N:>2} {tau:>4} {gc:>8.4f} {a:>4}/{B} {int(((P>0)&(sw[0]>1e-3)).sum()):>15} {int(((P>0)&osc).sum()):>16} {int(((P<0)&osc).sum()):>10}")
print(f"\nprediction from (loop sign, G_c) agrees with the long-run behaviour in {tot} of {n_all} runs")

relabelled ring vs original, max |x' - g x| after t = 40: 0.0e+00

 N  tau      G_c   agree  loop+ osc @500  loop+ osc @1000  loop- osc
----------------------------------------------------------------------
 1  0.0      inf   24/24               0                0          0
 1  1.0   2.2618   24/24               0                0          7
 2  0.0      inf   24/24               0                0          0
 2  1.0   1.6453   24/24               0                0          7
 3  0.0   2.0000   24/24               0                0          8
 3  1.0   1.3563   24/24               0                0          9
 4  0.0   1.4142   24/24               0                0         10
 4  1.0   1.2205   24/24               0                0          6
 5  0.0   1.2361   24/24               0                0         11
 5  1.0   1.1491   24/24               0                0         11
 6  0.0   1.1547   24/24               0                0         11
 6  1.0   1.1075   24/24          

The relabelling is exact to the last bit, and the prediction built from the loop sign and one location threshold agrees with every run. No positive loop oscillates at either horizon, whatever the gain or delay. A negative loop oscillates exactly when the gain crosses the phase-closure threshold. This is also the correction to the registry paper, read the right way round. The delayed positive ring could not hold a cycle, and the answer was in its address the whole time. Only a reader that looked at one location over one finite horizon could miss it.

---
## Step 5 — Any network: one bit per loop closed

A ring has one loop. A general network of readers has many. Take a signed graph: how many laws are there up to relabelling, and how much does a reader who has read the links $S$ know about which one it is? The claim predicts $2^{\beta_1}$ laws, with $\beta_1 = |E| - |V| + c$ the number of independent loops, and exactly $\beta_1(S)$ bits known. Every link set is checked exhaustively up to 10 links; the 3×3 grid and the Petersen graph use 150 random link sets each.

In [6]:
def cyc(n): return [(i, (i + 1) % n) for i in range(n)]
def complete(n): return list(itertools.combinations(range(n), 2))
petersen = [(i, (i + 1) % 5) for i in range(5)] + [(i, i + 5) for i in range(5)] + [(5 + i, 5 + (i + 2) % 5) for i in range(5)]
grid = [((r, c), (r, c + 1)) for r in range(3) for c in range(2)] + [((r, c), (r + 1, c)) for r in range(2) for c in range(3)]
graphs = {"path (tree)": [(i, i + 1) for i in range(4)], "ring C5": cyc(5), "ring C8": cyc(8),
          "theta": [(0, 1), (1, 2), (0, 3), (3, 2), (0, 4), (4, 2)], "two triangles": cyc(3) + [(3, 4), (4, 5), (5, 3)],
          "K4": complete(4), "K5": complete(5), "3x3 grid": grid, "Petersen": petersen}
import random
print(f"{'graph':>14} {'|V|':>4} {'|E|':>4} {'beta1':>6} {'laws':>6} {'2^beta1':>8}   reader knows exactly beta1(S) bits")
print("-" * 86)
for name, E in graphs.items():
    V = sorted({v for e in E for v in e}); m = len(E); b1 = beta1(E, V)
    signs = list(itertools.product((0, 1), repeat=m)); cls = [gauge_class(E, V, s) for s in signs]
    subsets = ([S for r in range(m + 1) for S in itertools.combinations(range(m), r)] if m <= 10 else
               [tuple(sorted(random.Random(i).sample(range(m), random.Random(1000 + i).randint(0, m)))) for i in range(150)])
    ok = sum(abs(mutual_info((tuple(s[j] for j in S), c) for s, c in zip(signs, cls)) - beta1([E[j] for j in S], V)) < 1e-9
             for S in subsets)
    print(f"{name:>14} {len(V):>4} {m:>4} {b1:>6} {len(set(cls)):>6} {2**b1:>8}   on {ok}/{len(subsets)} link sets")

         graph  |V|  |E|  beta1   laws  2^beta1   reader knows exactly beta1(S) bits
--------------------------------------------------------------------------------------
   path (tree)    5    4      0      1        1   on 16/16 link sets
       ring C5    5    5      1      2        2   on 32/32 link sets
       ring C8    8    8      1      2        2   on 256/256 link sets
         theta    5    6      2      4        4   on 64/64 link sets
 two triangles    6    6      2      4        4   on 64/64 link sets
            K4    4    6      3      8        8   on 64/64 link sets
            K5    5   10      6     64       64   on 1024/1024 link sets
      3x3 grid    9   12      4     16       16   on 150/150 link sets
      Petersen   10   15      6     64       64   on 150/150 link sets


Every graph has exactly $2^{\beta_1}$ laws. On every link set tested, the reader knows exactly $\beta_1(S)$ bits: one per independent loop inside what it has read. A tree has no loops, so it has one law and nothing to learn. $K_5$ has six loops' worth of law, and a reader learns it one closed loop at a time. The local→global gap therefore has an exact size, the cycle rank of what the reader has not closed, and an exact location, the open loops. It does not have an unknown law behind it.

The count $\beta_1 = |E| - |V| + c$ is the same formula as the corpus's capacity defect, $\beta_1(Q_r) = |E(Q_r)| - |V(Q_r)| + c(Q_r)$. The loop family's global bit (latch or clock) and the carry family's global count are the same kind of object: a cycle rank, a count of loops.

---
## What is proved, and what is not

- **[V]** On one bit, negation is the only fixed-point-free law, and it is the 2-cycle clock (Step 1).
- **[V]** For rings of 1 to 10 bits, all $2^N$ wirings reduce to exactly two laws, set by the loop bit; same-bit wirings are conjugate by relabelling, checked for every pattern up to $N = 8$; no relabelling crosses the loop bit (Step 2).
- **[V]** Any reader of a proper subset of links, or of one node for fewer than $N$ steps, knows 0 bits; closing the loop gives 1 bit, exhaustively for the ranges shown (Step 3).
- **[V-c]** The continuous rings agree in every run: $\tanh$ stages, one delay link, equal per-stage gains, 288 random configurations, two horizons, knife edges excluded. The general statement rests on monotone-systems theory for positive loops and Mallet-Paret–Sell for negative loops, which are cited here, not re-derived (Step 4).
- **[V]** In every graph and link set tested, the number of laws is $2^{\beta_1}$ and a reader knows exactly $\beta_1(S)$ bits (Step 5).
- **[H]** That "everything is a reader and being read" reaches beyond signed interaction networks, and that the corpus's capacity defect counts the same loops. These are readings, not results.

**What is not new.** Switching classes of signed graphs and their count $2^{\beta_1}$ are classical: Harary's balance theorem (1953) and Zaslavsky's theory of signed graphs (1982). "Monotone exactly when no loop is negative" is standard in monotone-systems theory (Smith 1995; Sontag 2007). These citations are from standard knowledge and were not re-checked here. The information statement $I(S;\text{law}) = \beta_1(S)$ follows in a few lines from the cycle space: a law is a coset of the cut space, and the only relabelling-invariant functions of the links in $S$ are products around loops inside $S$. It was not searched for in the literature.

## What it means

The law is the address: the loop bits, $\beta_1$ of them. The variable is the location: state, gain, delay. The only gap anywhere is a reader that has not yet closed a loop, and its size is exactly the number of loops left open. A system never has a gap about itself, because it closes its own loops on every circuit. That is what a circuit *is*. The registry's "empty slot" should be restated accordingly. Local→global is not a missing law. It is the cycle space, and every partial reader's gap is measured in it.

**Next test.** On the carry side, compute whether the capacity defect equals the number of loops a pulled-back reader cannot close. If it does, the two families share one invariant, not just one shape.